In [1]:
import os, random, argparse, itertools, numpy as np, scipy.io as sio
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score, silhouette_score
from sklearn.cluster import KMeans
from sklearn.preprocessing import scale
from _utils import MMDataset, clustering_acc, overall_performance_report

class DUCMME(nn.Module):
    def __init__(self, embed_dim=200, num_samples=10000, num_views=3, feature_dims=[1000, 1000, 500], hidden_dims=[512, 512, 512], n_clusters=10, alpha=1.0):
        super(DUCMME, self).__init__()
        self.embed_dim = embed_dim; self.num_samples = num_samples; self.num_views = num_views; self.feature_dims = feature_dims; self.hidden_dims = hidden_dims; self.n_clusters = n_clusters; self.alpha = alpha
        # 1. Multi-view Feature Extraction by Fusion-Net
        self.fusion_net_encoder = nn.ModuleList([nn.Sequential(nn.Linear(feature_dims[i], hidden_dims[i]), nn.BatchNorm1d(hidden_dims[i]), nn.ReLU(),
                                                               nn.Linear(hidden_dims[i], embed_dim)) for i in range(num_views)]) # encode each view
        self.fusion_net_mha = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=10, batch_first=True) # batch_first=True: (batch_size, seq_len, hidden_dim)
        self.fusion_net_linear = nn.Linear(3*embed_dim, embed_dim) # linear projection of the fused encoded features
        # 2. Uncertainty-Aware Reconstruction by Reconstruction-Net and Uncertainty-Net
        self.reconstruct_net_list = nn.ModuleList([nn.Sequential(nn.Linear(self.embed_dim, hidden_dims[i]), nn.BatchNorm1d(hidden_dims[i]), nn.ReLU(), 
                                                                 nn.Linear(hidden_dims[i], feature_dims[i])) for i in range(num_views)]) # reconstruct each view
        self.uncertainty_net_list = nn.ModuleList([nn.Sequential(nn.Linear(self.embed_dim, hidden_dims[i]), nn.BatchNorm1d(hidden_dims[i]), nn.ReLU(), 
                                                                 nn.Linear(hidden_dims[i], feature_dims[i])) for i in range(num_views)]) # predict uncertainty for each view
        # 3. Deep Embedding Clustering by DEC
        self._cluster_centers = nn.Parameter(torch.Tensor(self.n_clusters, self.embed_dim))
        nn.init.xavier_uniform_(self._cluster_centers.data)
        
    def forward_embedding(self, x):
        encoded_output_list = [self.fusion_net_encoder[i](x[i]) for i in range(self.num_views)] # encode each view
        encoded_output_list = torch.stack(encoded_output_list, dim=1) # stack the encoded features from all views, (batch_size, num_views, embed_dim)
        encoded_output_list, _ = self.fusion_net_mha(encoded_output_list, encoded_output_list, encoded_output_list) # fuse the encoded features from all views by a multihead attention, (batch_size, num_views, embed_dim)
        encoded_output_list = encoded_output_list.contiguous().view(encoded_output_list.shape[0], -1) # flatten the encoded features, (batch_size, num_views*embed_dim)
        embedding = self.fusion_net_linear(encoded_output_list) # linear projection of the fused encoded features
        return embedding # get the embedding of the latent space H, (batch_size, embed_dim)

    def forward_uncertainty_aware_reconstruction(self, x):
        embedding = self.forward_embedding(x) # shape: [batch_size, embed_dim]
        reconstructions = [self.reconstruct_net_list[i](embedding) for i in range(self.num_views)] # reconstruct each view
        uncertainties = [self.uncertainty_net_list[i](embedding) for i in range(self.num_views)] # predict uncertainty for each view
        return reconstructions, uncertainties
        
    def forward_similarity_matrix_q(self, x): # calculate the similarity matrix q using t-distribution
        embedding = self.forward_embedding(x) # shape: [batch_size, embed_dim]
        q = 1.0 / (1.0 + torch.sum((embedding.unsqueeze(1) - self._cluster_centers) ** 2, dim=2) / self.alpha) # shape: [batch_size, n_clusters]
        q = q ** ((self.alpha + 1.0) / 2.0) # , shape: [batch_size, n_clusters]
        q = q / torch.sum(q, dim=1, keepdim=True) # Normalize q to sum to 1 across clusters, shape: [batch_size, n_clusters]
        return q, embedding # q can be regarded as the probability of the sample belonging to each cluster
    
    @property
    def cluster_centers(self):
        return self._cluster_centers.data.detach().cpu().numpy() # shape: (n_clusters, embed_dim)
    
    @cluster_centers.setter
    def cluster_centers(self, centers): # shape: (n_clusters, embed_dim)
        centers = torch.tensor(centers, dtype=torch.float32, device=self._cluster_centers.device)
        self._cluster_centers.data.copy_(centers) # copy the cluster centers to the model, set the cluster centers to the new cluster centers
        
    @staticmethod
    def target_distribution(q):
        weight = q ** 2 / torch.sum(q, dim=0) # shape: [batch_size, n_clusters]
        p = weight / torch.sum(weight, dim=1, keepdim=True) # Normalize p to sum to 1 across clusters, shape: [batch_size, n_clusters]
        return p.clone().detach()
    
    def reconstruction_loss(self, x):
        x = [x[i] + 0.1 * torch.randn_like(x[i]) for i in range(self.num_views)] # add noise to the input data
        x_rec, _ = self.forward_uncertainty_aware_reconstruction(x) # reconstruct each view and predict uncertainty
        return sum([F.mse_loss(x_rec[v], x[v], reduction='mean') for v in range(self.num_views)]) # sum the losses from all views
    
    def uncertainty_aware_reconstruction_loss(self, x):
        x = [x[i] + 0.1 * torch.randn_like(x[i]) for i in range(self.num_views)] # add noise to the input data
        x_rec, log_sigma_2 = self.forward_uncertainty_aware_reconstruction(x) # reconstruct each view and predict uncertainty
        # Clip log_sigma_2 to prevent numerical instability (exp(-log_sigma_2) overflow/underflow)
        # Clamp to reasonable range: -10 to 10, which corresponds to sigma^2 from exp(-10) to exp(10)
        log_sigma_2 = [torch.clamp(log_sigma_2[v], min=-10.0, max=10.0) for v in range(self.num_views)] # shape: [num_views, batch_size, feature_dim] for numerically stable computation
        return sum([0.5 * torch.mean((x_rec[v] - x[v])**2 * torch.exp(-log_sigma_2[v]) + log_sigma_2[v]) for v in range(self.num_views)]) # uncertainty is equal to log_sigma_2
    
    def clustering_loss(self, x, p):
        x = [x[i] + 0.1 * torch.randn_like(x[i]) for i in range(self.num_views)] # add noise to the input data
        q, _ = self.forward_similarity_matrix_q(x) # shape: [batch_size, n_clusters]
        return F.kl_div(q.log(), p, reduction='batchmean') # shape: ()

In [4]:
random.seed(0); np.random.seed(0); torch.manual_seed(0); torch.cuda.manual_seed_all(0) # Set random seed for reproducibility
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dataset = MMDataset('./data/data_sc_multiomics/NEAT/', concat_data=False)
data = [x.clone().to(device) for x in dataset.X]; label = dataset.Y.clone().numpy()
data_views = dataset.data_views; data_samples = dataset.data_samples; data_features = dataset.data_features; data_categories = dataset.categories
label_dict = dataset.get_label_to_name()

dataloader = torch.utils.data.DataLoader(dataset, batch_size=2000, shuffle=True)
model = DUCMME(embed_dim=20, feature_dims=data_features, num_views=data_views, hidden_dims=[512, 512, 512], num_samples=data_samples, n_clusters=data_categories, alpha=1.0).to(device)
## === Stage 1: Uncertainty-Aware Reconstruction Pretraining ===
print("\n=== Stage 1: Uncertainty-Aware Reconstruction Pretraining ===")
print("Basic reconstruction training...")
optimizer = torch.optim.Adam(model.parameters(), lr=5e-3)
losses_convergence_plot_reconstruction = []
for epoch in range(200):
    model.train()
    losses = []
    for batch_idx, (x, y, idx) in enumerate(dataloader):
        x = [x.to(device) for x in x]
        optimizer.zero_grad()
        loss = model.reconstruction_loss(x)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    losses_convergence_plot_reconstruction.append(np.mean(losses))
    print(f'Epoch {epoch} completed. Average Loss: {np.mean(losses):.4f}')
embedding = model.forward_embedding(data).detach().cpu().numpy() # shape: [num_samples, embed_dim]
kmeans = KMeans(n_clusters=data_categories, n_init=20, random_state=0)
preds = kmeans.fit_predict(embedding)
acc_val = clustering_acc(label, preds)
nmi_val = normalized_mutual_info_score(label, preds)
asw_val = silhouette_score(embedding, preds)
ari_val = adjusted_rand_score(label, preds)
print(f"Pretraining completed. ACC: {acc_val:.4f}, NMI: {nmi_val:.4f}, ASW: {asw_val:.4f}, ARI: {ari_val:.4f}")
print("Uncertainty-aware reconstruction finetuning...")
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
losses_convergence_plot_reconstruction_finetuning = []
for epoch in range(100):
    model.train()
    losses = []
    for batch_idx, (x, y, idx) in enumerate(dataloader):
        x = [x.to(device) for x in x]
        optimizer.zero_grad()
        loss = model.uncertainty_aware_reconstruction_loss(x)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    losses_convergence_plot_reconstruction_finetuning.append(np.mean(losses))
    print(f'Epoch {epoch} completed. Average Loss: {np.mean(losses):.4f}')
embedding = model.forward_embedding(data).detach().cpu().numpy() # shape: [num_samples, embed_dim]
kmeans = KMeans(n_clusters=data_categories, n_init=20, random_state=0)
preds = kmeans.fit_predict(embedding)
acc_val = clustering_acc(label, preds)
nmi_val = normalized_mutual_info_score(label, preds)
asw_val = silhouette_score(embedding, preds)
ari_val = adjusted_rand_score(label, preds)
print(f"Finetuning completed. ACC: {acc_val:.4f}, NMI: {nmi_val:.4f}, ASW: {asw_val:.4f}, ARI: {ari_val:.4f}")

## === Stage 2: Deep Embedding Clustering by DEC ===
print("\n=== Stage 2: Deep Embedding Clustering ===")
print("Cluster center initialization...")
model.eval()
embedding = model.forward_embedding(data).detach().cpu().numpy() # shape: [num_samples, embed_dim]
kmeans = KMeans(n_clusters=data_categories, n_init=20, random_state=0)
preds = kmeans.fit_predict(embedding)
acc_val = clustering_acc(label, preds)
nmi_val = normalized_mutual_info_score(label, preds)
asw_val = silhouette_score(embedding, preds)
ari_val = adjusted_rand_score(label, preds)
print(f"Cluster center initialization completed. ACC: {acc_val:.4f}, NMI: {nmi_val:.4f}, ASW: {asw_val:.4f}, ARI: {ari_val:.4f}")
model.cluster_centers = kmeans.cluster_centers_ # shape: (n_clusters, embed_dim)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.9)
losses = []
acc_convergence_plot = []
nmi_convergence_plot = []
asw_convergence_plot = []
ari_convergence_plot = []
embeddings_dict = {}
preds_dict = {}
for epoch in range(100):
    # Update target distribution periodically
    if epoch % 1 == 0:
        model.eval()
        with torch.no_grad():
            q, embedding = model.forward_similarity_matrix_q(data)
            p = model.target_distribution(q)
        y_pred = torch.argmax(q, dim=1).cpu().numpy()
        acc_val = clustering_acc(label, y_pred)
        nmi_val = normalized_mutual_info_score(label, y_pred)
        asw_val = silhouette_score(embedding.cpu().numpy(), y_pred) if np.unique(y_pred).shape[0] > 1 else 0.0
        ari_val = adjusted_rand_score(label, y_pred)
        acc_convergence_plot.append(acc_val)
        nmi_convergence_plot.append(nmi_val)
        asw_convergence_plot.append(asw_val)
        ari_convergence_plot.append(ari_val)
        embeddings_dict[epoch] = embedding
        preds_dict[epoch] = y_pred
        if epoch == 0:
            delta_label = 1.0
            y_pred_last = y_pred.copy()
            print(f'[Epoch {epoch}] ACC: {acc_val:.4f}, NMI: {nmi_val:.4f}, ASW: {asw_val:.4f}, ARI: {ari_val:.4f}, Delta: {delta_label:.4f}')
        else:
            delta_label = np.sum(y_pred != y_pred_last).astype(np.float32) / y_pred.shape[0]
            y_pred_last = y_pred.copy()
            print(f'[Epoch {epoch}] ACC: {acc_val:.4f}, NMI: {nmi_val:.4f}, ASW: {asw_val:.4f}, ARI: {ari_val:.4f}, Delta: {delta_label:.4f}')
            if delta_label < 0.001:
                convergence_epoch = epoch
                print('Converged, stopping training...')
                # break
    # Training based on the target distribution that is updated periodically
    model.train()
    losses = []
    for batch_idx, (x, y, idx) in enumerate(dataloader):
        x = [x.to(device) for x in x]
        optimizer.zero_grad()
        loss = model.clustering_loss(x, p[idx])
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    scheduler.step()
    print(f'Epoch {epoch} completed. Average Loss: {np.mean(losses):.4f}')
print(f'Final ACC: {clustering_acc(label, y_pred):.4f}')

modality_rna shape: (8472, 100)
modality_protein shape: (8472, 7)
modality_atac shape: (8472, 100)

=== Stage 1: Uncertainty-Aware Reconstruction Pretraining ===
Basic reconstruction training...
Epoch 0 completed. Average Loss: 0.8114
Epoch 1 completed. Average Loss: 0.3804
Epoch 2 completed. Average Loss: 0.2279
Epoch 3 completed. Average Loss: 0.1602
Epoch 4 completed. Average Loss: 0.1199
Epoch 5 completed. Average Loss: 0.0961
Epoch 6 completed. Average Loss: 0.0788
Epoch 7 completed. Average Loss: 0.0667
Epoch 8 completed. Average Loss: 0.0593
Epoch 9 completed. Average Loss: 0.0543
Epoch 10 completed. Average Loss: 0.0509
Epoch 11 completed. Average Loss: 0.0485
Epoch 12 completed. Average Loss: 0.0472
Epoch 13 completed. Average Loss: 0.0460
Epoch 14 completed. Average Loss: 0.0451
Epoch 15 completed. Average Loss: 0.0445
Epoch 16 completed. Average Loss: 0.0439
Epoch 17 completed. Average Loss: 0.0434
Epoch 18 completed. Average Loss: 0.0432
Epoch 19 completed. Average Loss: 0.

In [ ]:
# [Epoch 34] ACC: 0.4510, NMI: 0.1526, ASW: 0.3894, ARI: 0.1273, Delta: 0.0155
# Epoch 34 completed. Average Loss: 0.0787
# [Epoch 35] ACC: 0.4516, NMI: 0.1644, ASW: 0.3944, ARI: 0.1223, Delta: 0.0274
# Epoch 35 completed. Average Loss: 0.0700
# [Epoch 36] ACC: 0.4536, NMI: 0.1746, ASW: 0.4055, ARI: 0.1233, Delta: 0.0166
# Epoch 36 completed. Average Loss: 0.0662
# [Epoch 37] ACC: 0.4579, NMI: 0.1852, ASW: 0.4189, ARI: 0.1291, Delta: 0.0143
# Epoch 37 completed. Average Loss: 0.0600
# [Epoch 38] ACC: 0.4612, NMI: 0.1944, ASW: 0.4340, ARI: 0.1343, Delta: 0.0127